# 01 · Acquire market prices

> **Run order.** This notebook is step 1 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Loads the company list and five years of daily OHLCV into Postgres.

**The trap this step exists to handle:** while a market is open — and
occasionally on a *completed* session — Yahoo returns a row with Open/High/Low/
Volume populated and `Close` = `NaN`. `df.iloc[-1]["Close"]` is then NaN in
production, intermittently. `drop_partial_bar()` removes any row without a close,
and a regression test pins the behaviour.

## Corpus definition

Scope changes in exactly one place: `configs/companies.yaml`.

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from analyst.corpus import load_corpus

companies = load_corpus()
pd.DataFrame([c.model_dump() for c in companies])

## What a raw Yahoo response looks like

Note the final row: this is the bug, visible.

In [ ]:
import yfinance as yf
from analyst.prices import drop_partial_bar, to_rows

raw = yf.Ticker("RELIANCE.NS").history(period="1mo", auto_adjust=False)
print(f"rows returned: {len(raw)}")
raw.tail(3)[["Open", "High", "Low", "Close", "Volume"]]

In [ ]:
clean = drop_partial_bar(raw)
print(f"before: {len(raw)} rows   after: {len(clean)} rows   dropped: {len(raw) - len(clean)}")
print("\nA dropped row is one with no close price - an incomplete session, or a vendor gap.")
clean.tail(2)[["Open", "Close", "Volume"]]

## Ingest everything

Idempotent: `ON CONFLICT (ticker, trade_date) DO NOTHING`. Re-running tomorrow adds tomorrow's bar and leaves history untouched.

In [ ]:
from sqlalchemy.dialects.postgresql import insert
from tenacity import retry, stop_after_attempt, wait_exponential

from analyst.db import session_scope
from analyst.models import Company, Price

PERIOD, CHUNK = "5y", 500

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_history(symbol: str):
    df = yf.Ticker(symbol).history(period=PERIOD, auto_adjust=False)
    if df.empty:
        raise ValueError(f"no price rows returned for {symbol}")
    return df

with session_scope() as s:
    for c in companies:
        s.execute(
            insert(Company)
            .values(ticker=c.ticker, name=c.name, sector=c.sector, yf_symbol=c.yf_symbol)
            .on_conflict_do_update(
                index_elements=[Company.ticker],
                set_={"name": c.name, "sector": c.sector, "yf_symbol": c.yf_symbol},
            )
        )

summary = []
for c in companies:
    raw = fetch_history(c.yf_symbol)
    clean = drop_partial_bar(raw)
    rows = to_rows(clean, c.ticker)
    with session_scope() as s:
        for i in range(0, len(rows), CHUNK):
            s.execute(
                insert(Price)
                .values([r.model_dump() for r in rows[i : i + CHUNK]])
                .on_conflict_do_nothing(constraint="uq_prices_ticker_date")
            )
    summary.append({
        "ticker": c.ticker, "rows": len(rows),
        "dropped": len(raw) - len(clean),
        "first": rows[0].trade_date, "last": rows[-1].trade_date,
    })

df = pd.DataFrame(summary)
print(f"total price rows ingested: {df['rows'].sum():,}")
df